# MEG Data Preprocessing and Epoching

##  Overview

This notebook contains the preprocessing pipeline used to convert the
raw MEG recordings into epoched MNE-Python datasets for the subsequent
decoding and replay analyses.

The raw data are organized by participant. For each participant, the
pipeline identifies matching `.mat` and `.evt` files corresponding to
individual recording runs.

Each matched MAT/EVT pair is processed using the `process_recording()`
function. The resulting epochs from all runs belonging to the same
participant are then concatenated into a single dataset.

The final epoched datasets are saved in FIF format and assigned
sequential subject identifiers (`Sub1`, `Sub2`, etc.).

In [1]:
#importing necessary libraries
from pathlib import Path
import numpy as np
import pandas as pd
import mne
from scipy.io import loadmat
from sklearn.decomposition import PCA

##  Processing individual recordings

The `process_recording()` function converts one raw recording and its
corresponding event file into an MNE-Python `Epochs` object.

For each recording, the function performs the following steps:

1. **Load the raw MATLAB data**

   The MATLAB file contains the three spatial components of the signal
   (`x`, `y`, and `z`) for each MEG node, together with the recording
   time vector.

2. **Combine the three spatial components**

   For each MEG node, the three components are combined using
   principal component analysis (PCA).

   For each node, the signal is represented as:

   `x, y, z → PCA → first principal component`

   The first principal component provides a single time series capturing
   the dominant variance across the three spatial components.

   Nodes containing NaN values are skipped.

3. **Load the event information**

   The corresponding `.evt` file contains the event onset times and
   trigger codes.

   Event onsets are taken from the `Tsec` column, while event identities
   are taken from the `TriNo` column.

4. **Create an MNE Raw object**

   The PCA-reduced signals are converted into an MNE `RawArray`, with
   one channel for each MEG node.

5. **Map experimental events**

   The original trigger codes are mapped to four event classes:

   | Trigger | Event |
   |---------|-------|
   | 9       | stay left |
   | 13      | stay right |
   | 17      | shift left |
   | 21      | shift right |

6. **Extract epochs**

   Continuous data are segmented around each event from **−0.5 to
   +1.0 seconds**.

   Baseline correction is performed using the **−0.5 to 0 second**
   pre-event interval.

The function returns an MNE `Epochs` object containing the processed
trials for that recording.

In [ ]:
#Main function to process the recordings and extract epochs
# =========================================================
# PROCESSING FUNCTION 
# =========================================================
def process_recording(mat_file, evt_file,
                      tmin=-0.5,
                      tmax=1,
                      baseline=(-0.5, 0)):

    data = loadmat(mat_file)

    x = data['sigx_tot'].squeeze()
    y = data['sigy_tot'].squeeze()
    z = data['sigz_tot'].squeeze()
    time = data['time_ica'].squeeze()

    n_nodes = x.shape[0]
    n_timepoints = x.shape[1]

    combined_signal = np.zeros((n_nodes, n_timepoints))

    for i in range(n_nodes):
        data_node = np.vstack([x[i], y[i], z[i]]).T


        pca = PCA(n_components=1)
        combined_signal[i] = pca.fit_transform(data_node)[:, 0]

    triggers = pd.read_csv(evt_file, sep=r"\s+", engine="python")

    onsets = triggers.Tsec.values
    duration = np.ones(len(onsets))
    descr = triggers.TriNo.astype(str).values

    annotations = mne.Annotations(
        onset=onsets,
        duration=duration,
        description=descr
    )

    dt = np.median(np.diff(time))
    sfreq = 1 / dt

    info = mne.create_info(
        ch_names=[f"node{i}" for i in range(n_nodes)],
        sfreq=sfreq,
        ch_types='misc'
    )

    raw = mne.io.RawArray(combined_signal, info)
    raw.set_annotations(annotations)

    # =====================================================
    # FIXED EVENT MAPPING
    # =====================================================
    FIXED_EVENT_ID = {
        "9": 1,
        "13": 2,
        "17": 3,
        "21": 4
    }
#     "stay_left": 9,
#     "stay_right": 13,
#     "shift_left": 17,
#     "shift_right": 21,
    events, event_id = mne.events_from_annotations(
        raw,
        event_id=FIXED_EVENT_ID
    )

    epochs = mne.Epochs(
        raw,
        events,
        event_id=FIXED_EVENT_ID,
        tmin=tmin,
        tmax=tmax,
        baseline=baseline,
        preload=True
    )

    return epochs



### Processing steps

For each participant, the pipeline:

1. Identifies the participant folder.
2. Finds all `.mat` and `.evt` recording files.
3. Matches MAT and EVT files based on their filename.
4. Processes each matched recording using `process_recording()`.
5. Concatenates all runs from the same participant.
6. Saves the resulting epochs as an MNE `.fif` file.

The processing output printed below provides a record of which
participants and recording runs were successfully processed and
whether any recordings were skipped.

In [ ]:

# =========================================================
# PATHS — CHANGE THESE TO YOUR LOCAL DIRECTORIES
# =========================================================

base_path = Path("/path/to/your/data")

output_path = Path("/path/to/your/epoched_data")

output_path.mkdir(parents=True, exist_ok=True)

participant_dirs = sorted([p for p in base_path.iterdir() if p.is_dir()])

sub_counter = 1

for participant_path in participant_dirs:

    participant_id = participant_path.name

    print(f"\n==============================")
    print(f"Processing {participant_id}")
    print(f"==============================")

    mat_files = sorted(participant_path.glob("*.mat"))
    evt_files = sorted(participant_path.glob("*.evt"))

    mat_dict = {f.stem: f for f in mat_files}
    evt_dict = {f.stem: f for f in evt_files}

    common_keys = sorted(set(mat_dict.keys()) & set(evt_dict.keys()))

    if len(common_keys) == 0:
        print("No matching MAT/EVT pairs found, skipping.")
        continue

    participant_epochs = []

    for key in common_keys:

        print(f"  Run: {key}")

        epochs = process_recording(
            mat_file=str(mat_dict[key]),
            evt_file=str(evt_dict[key])
        )

        participant_epochs.append(epochs)

    if len(participant_epochs) == 0:
        continue

    # merge runs
    epochs_all = mne.concatenate_epochs(participant_epochs)

    # save with Sub numbering
    #save_name = f"Sub{sub_counter}_epo.fif"
    #save_path = output_path / save_name

    #epochs_all.save(save_path, overwrite=True)

    #print(f"Saved: {save_name}")

    sub_counter += 1
    
    

FileNotFoundError: [Errno 2] No such file or directory: '/home/uranus/Scrivania/MEG_replay/codes/notebooks/data'